In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
 
TARGET_COLS = ["step", "normalized_location_entropy", "totalSleep"]
TIME_COLS = ["startSleepTime"]


# ---------------------------------------------------------------------------
# 0. 시각(time) 컬럼 전처리
# ---------------------------------------------------------------------------
def add_decimal_sleep_hour(df: pd.DataFrame, col: str = "startSleepTime") -> pd.DataFrame:
    """
    'startSleepTime' 같은 시각 문자열에서 날짜를 버리고 시각만 남겨
    다음 두 컬럼을 추가:
      - {col}_time     : datetime.time 객체 (시:분만 남은 순수 시각)
      - {col}_decimal  : 0~24 범위의 float 시각 (원본 24시간제 그대로)
 
    통계/카테고리화는 이 24시간제 값을 그대로 사용한다.
    자정 근처에서 값이 끊겨 보이는 문제는 시각화(histogram) 단계에서만
    별도로 처리한다 (make_noon_centered_plot_values 참고).
    """
    df = df.copy()
    parsed = pd.to_datetime(df[col], format="mixed", errors="coerce")
    df[f"{col}_time"] = parsed.dt.time  # 날짜 제거, 시각만 남김
    df[f"{col}_decimal"] = parsed.dt.hour + parsed.dt.minute / 60
 
    n_failed = parsed.isna().sum() - df[col].isna().sum()
    if n_failed > 0:
        print(f"[경고] '{col}' 중 {n_failed}개 값을 datetime으로 파싱하지 못했습니다.")
 
    return df
 
 
def make_noon_centered_plot_values(hour_decimal: pd.Series) -> pd.Series:
    """
    24시간제(0~24) 시각 값을 '표시용'으로만 변환.
    0~11시대 값에 +24를 더해 12~36 범위로 만들면,
    x축이 낮12시 -> 자정(24) -> 다음날 낮12시(36) 순서로 이어져서
    자정을 넘나드는 분포가 끊기지 않고 하나로 보인다.
 
    통계 계산에는 쓰지 말고, 오직 히스토그램/박스플롯 그릴 때만 사용할 것.
    """
    return hour_decimal.apply(lambda h: h + 24 if h < 12 else h)
 
 
def _format_noon_centered_ticks(ax):
    """make_noon_centered_plot_values로 그린 축의 눈금을 다시 0~24시 라벨로 표시."""
    tick_positions = [12, 15, 18, 21, 24, 27, 30, 33, 36]
    tick_labels = ["12", "15", "18", "21", "0", "3", "6", "9", "12"]
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)


# ---------------------------------------------------------------------------
# 1. EDA
# ---------------------------------------------------------------------------
def run_eda(
        df: pd.DataFrame,
        cols: list[str] = TARGET_COLS,
        time_cols: list[str] = TIME_COLS,
        ):
    """
    수치형/시각형 컬럼을 자동 구분해서 각각에 맞는 통계와 시각화를 수행.
    시각 컬럼(time_cols)은 통계 계산엔 원본 24시간제 값을 쓰고,
    히스토그램/박스플롯을 그릴 때만 표시용으로 정오~정오 축으로 wrap한다.
    """
    df = df.copy()
 
    numeric_cols = []
    for col in cols:
        if col in time_cols:
            df = add_decimal_sleep_hour(df, col)
            numeric_cols.append(f"{col}_decimal")
        else:
            numeric_cols.append(col)
 
    print("=" * 60)
    print("기초 통계량 (describe)")
    print("=" * 60)
    print(df[numeric_cols].describe().T)
 
    print("\n결측치 개수")
    print(df[numeric_cols].isna().sum())
 
    n = len(numeric_cols)
    fig, axes = plt.subplots(n, 2, figsize=(12, 4 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
 
    for i, col in enumerate(numeric_cols):
        data = df[col].dropna()
        is_time_col = col.endswith("_decimal") and col.replace("_decimal", "") in time_cols
 
        if is_time_col:
            plot_data = make_noon_centered_plot_values(data)
        else:
            plot_data = data
 
        sns.histplot(plot_data, kde=True, ax=axes[i, 0], bins=30)
        axes[i, 0].set_title(f"{col} - Histogram + KDE")
        axes[i, 0].axvline(plot_data.mean(), color="red", linestyle="--", label="mean")
        axes[i, 0].axvline(plot_data.median(), color="green", linestyle="--", label="median")
        axes[i, 0].legend()
        if is_time_col:
            _format_noon_centered_ticks(axes[i, 0])
            axes[i, 0].set_xlabel("시각 (0~24시, 자정이 가운데)")
 
        sns.boxplot(x=plot_data, ax=axes[i, 1])
        axes[i, 1].set_title(f"{col} - Boxplot")
        if is_time_col:
            _format_noon_centered_ticks(axes[i, 1])
            axes[i, 1].set_xlabel("시각 (0~24시, 자정이 가운데)")
 
    plt.tight_layout()
    plt.savefig("eda_distribution.png", dpi=150)
    plt.show()
    print("\n분포 이미지 저장 완료: eda_distribution.png")
 
    return df  # decimal 컬럼이 추가된 df 반환 (이후 categorize에 재사용 가능)
 

df = pd.read_csv("../data/merged_219.csv")
df["startSleepTime"] = pd.to_datetime(df["startSleepTime"], format="mixed", errors="coerce")

df = run_eda(df, TARGET_COLS)

NameError: name 'time_cols' is not defined